In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.sql("CREATE CATALOG IF NOT EXISTS company_ro")
spark.sql("CREATE SCHEMA IF NOT EXISTS company_ro.gold")

In [0]:
%sql
CREATE OR REPLACE MATERIALIZED VIEW company_ro.gold.top_companies_by_year
AS
WITH ranked_companies AS (
  SELECT
    an,
    cui,
    denumire,
    judet,
    localitate,
    cod_caen_mfinante,
    denumire_caen,
    grupa_caen,
    cifra_afaceri,
    profit_net,
    pierdere_neta,
    nr_mediu_salariati,
    datorii,
    capitaluri_proprii,
    ROW_NUMBER() OVER (PARTITION BY an ORDER BY cifra_afaceri DESC NULLS LAST) AS rank_by_turnover,
    ROW_NUMBER() OVER (PARTITION BY an ORDER BY profit_net DESC NULLS LAST) AS rank_by_profit,
    ROW_NUMBER() OVER (PARTITION BY an ORDER BY nr_mediu_salariati DESC NULLS LAST) AS rank_by_employees
  FROM company_ro.gold.company_financial_summary
)
SELECT *
FROM ranked_companies
WHERE rank_by_turnover <= 100
   OR rank_by_profit <= 100
   OR rank_by_employees <= 100

In [0]:
%sql
-- To manually refresh the materialized view:
-- REFRESH MATERIALIZED VIEW company_ro.gold.top_companies_by_year

In [0]:
display(spark.sql("""
SELECT
  an,
  COUNT(*) AS rows,
  MIN(rank_by_turnover) AS best_turnover_rank,
  MAX(rank_by_turnover) AS worst_turnover_rank
FROM company_ro.gold.top_companies_by_year
GROUP BY an
ORDER BY an
"""))